# Member 4 — LightGBM
**Algorithm:** LightGBM — Light Gradient Boosting Machine (Supervised)  
**Dataset:** NSL-KDD (`nsl_kdd_dataset.csv`) — 41 pre-scaled features  
**Training data:** SMOTE-balanced (from shared preprocessing)

### How LightGBM Works
LightGBM uses **leaf-wise** tree growth (vs level-wise in XGBoost) which
tends to find better splits and converge faster. It also uses histogram-based
splitting for speed. It supports **DART** (dropout for additive regression trees)
boosting which further reduces overfitting.

### Hyperparameter Tuning
- **Phase 1:** `RandomizedSearchCV` — wide search (50 iterations, 5-fold CV)
- **Phase 2:** `GridSearchCV` — fine-grained search (5-fold CV)
- Scoring: **F1**

## 1. Import Libraries

In [ ]:
# !pip install lightgbm shap

import pickle, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_val_score
)
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, roc_curve
)
from scipy.stats import uniform, randint

try:
    import shap
    SHAP_OK = True
    print('SHAP available.')
except ImportError:
    SHAP_OK = False

print('Libraries loaded.')

## 2. Load Preprocessed Data

In [ ]:
with open('../data/processed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train      = data['X_train']
X_test       = data['X_test']
y_train      = data['y_train']
y_test       = data['y_test']
feature_cols = data['feature_cols']

print(f'X_train (SMOTE) : {X_train.shape} | normal={np.sum(y_train==0)}, attack={np.sum(y_train==1)}')
print(f'X_test  (real)  : {X_test.shape}')

## 3. Phase 1 — RandomizedSearchCV (Wide Search)

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'n_estimators'      : randint(100, 800),
    'num_leaves'        : randint(20, 150),
    'max_depth'         : [-1, 5, 10, 15, 20],
    'learning_rate'     : uniform(0.01, 0.3),
    'subsample'         : uniform(0.6, 0.4),
    'colsample_bytree'  : uniform(0.5, 0.5),
    'min_child_samples' : randint(5, 50),
    'reg_alpha'         : uniform(0, 1.0),
    'reg_lambda'        : uniform(0, 1.0),
    'min_split_gain'    : uniform(0, 0.5),
    'boosting_type'     : ['gbdt', 'dart'],
}

lgbm_rand = RandomizedSearchCV(
    LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1),
    param_distributions=param_dist,
    n_iter=50,
    cv=cv5,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
lgbm_rand.fit(X_train, y_train)

print(f'\nBest params (random): {lgbm_rand.best_params_}')
print(f'Best CV F1 (random) : {lgbm_rand.best_score_:.4f}')

## 4. Phase 2 — GridSearchCV (Fine-Grained)

In [ ]:
bp = lgbm_rand.best_params_

def rnd(v, step, n=1, low=0.001):
    return sorted(set([round(max(low, v + i*step), 4) for i in range(-n, n+1)]))

def int_neighbors(v, step=20, n=1, low=1):
    return sorted(set([max(low, v + i*step) for i in range(-n, n+1)]))

param_grid = {
    'n_estimators' : int_neighbors(bp['n_estimators'], 50, n=1, low=50),   # 3 values
    'num_leaves'   : int_neighbors(bp['num_leaves'], 10, n=1, low=10),     # 3 values
    'learning_rate': rnd(bp['learning_rate'], 0.02, n=1),                  # 3 values
    'subsample'    : rnd(bp['subsample'], 0.05, n=1, low=0.5),             # 3 values
}

lgbm_grid = GridSearchCV(
    LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1),
    param_grid=param_grid,
    cv=cv5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
lgbm_grid.fit(X_train, y_train)

best_params = lgbm_grid.best_params_
print(f'\nBest params (grid) : {best_params}')
print(f'Best CV F1 (grid)  : {lgbm_grid.best_score_:.4f}')

## 5. Train Final LightGBM Model

In [ ]:
lgbm_final = LGBMClassifier(
    **best_params,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(period=-1)]
)
print('Final LightGBM trained.')
print(f'Best iteration: {lgbm_final.best_iteration_}')

## 6. 5-Fold Cross-Validation

In [ ]:
lgbm_cv = LGBMClassifier(**best_params, class_weight='balanced',
                          random_state=42, n_jobs=-1, verbose=-1)

cv_f1  = cross_val_score(lgbm_cv, X_train, y_train, cv=cv5, scoring='f1',       n_jobs=-1)
cv_acc = cross_val_score(lgbm_cv, X_train, y_train, cv=cv5, scoring='accuracy', n_jobs=-1)
cv_auc = cross_val_score(lgbm_cv, X_train, y_train, cv=cv5, scoring='roc_auc',  n_jobs=-1)

print('=== 5-Fold Cross-Validation ===')
print(f'F1       : {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}')
print(f'Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'ROC-AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, scores, name, color in zip(
    axes, [cv_f1, cv_acc, cv_auc],
    ['F1','Accuracy','ROC-AUC'], ['#8e44ad','#16a085','#d35400']
):
    ax.plot(range(1,6), scores, 'o-', color=color, lw=2, markersize=7)
    ax.axhline(scores.mean(), linestyle='--', color=color, alpha=0.5,
               label=f'Mean={scores.mean():.4f}')
    ax.set_title(f'CV {name}')
    ax.set_xlabel('Fold')
    ax.legend(fontsize=9)
    ax.set_xticks(range(1,6))
plt.suptitle('LightGBM — 5-Fold CV Scores', fontsize=13)
plt.tight_layout()
plt.savefig('cv_scores_lgbm.png', dpi=150)
plt.show()

## 7. Test Set Evaluation

In [ ]:
y_pred      = lgbm_final.predict(X_test)
y_pred_prob = lgbm_final.predict_proba(X_test)[:, 1]

print('=== Test Set Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Normal','Attack']))

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
roc_auc   = roc_auc_score(y_test, y_pred_prob)

print(f'Accuracy  : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'ROC-AUC   : {roc_auc:.4f}')

## 8. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Normal','Attack']).plot(
    cmap='Purples', ax=axes[0])
axes[0].set_title('LightGBM — Confusion Matrix')

fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='#8e44ad', lw=2, label=f'LightGBM (AUC={roc_auc:.4f})')
axes[1].plot([0,1],[0,1],'--', color='navy', lw=1, label='Random')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('LightGBM — ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.savefig('cm_roc_lgbm.png', dpi=150)
plt.show()

## 9. Feature Importance

In [ ]:
feat_df = pd.DataFrame({
    'Feature'   : feature_cols,
    'Importance': lgbm_final.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=feat_df.head(15), x='Importance', y='Feature', palette='Purples_r')
plt.title('LightGBM — Top 15 Feature Importances (split gain)')
plt.tight_layout()
plt.savefig('feature_importance_lgbm.png', dpi=150)
plt.show()
print(feat_df.head(15).to_string(index=False))

## 10. SHAP Values

In [ ]:
if SHAP_OK:
    n_shap   = min(1000, len(X_test))
    idx_shap = np.random.RandomState(42).choice(len(X_test), n_shap, replace=False)
    X_shap   = X_test[idx_shap]
    explainer   = shap.TreeExplainer(lgbm_final)
    shap_values = explainer.shap_values(X_shap)
    # LightGBM returns list [class0, class1]; take class1 (attack)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv, X_shap, feature_names=feature_cols, show=False)
    plt.title('LightGBM — SHAP Summary Plot (Attack class)')
    plt.tight_layout()
    plt.savefig('shap_lgbm.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('pip install shap')

## 11. LightGBM Training Curve

In [ ]:
# Retrain with full logging to plot learning curve
evals_result = {}
lgbm_log = LGBMClassifier(
    **best_params,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_log.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    eval_metric='binary_logloss',
    callbacks=[lgb.record_evaluation(evals_result),
               lgb.log_evaluation(period=-1)]
)

train_loss = evals_result['training']['binary_logloss']
test_loss  = evals_result['valid_1']['binary_logloss']

plt.figure(figsize=(8, 4))
plt.plot(train_loss, label='Train Loss', color='steelblue', lw=1.5)
plt.plot(test_loss,  label='Test Loss',  color='crimson',   lw=1.5)
plt.xlabel('Iteration')
plt.ylabel('Binary Log Loss')
plt.title('LightGBM — Learning Curve')
plt.legend()
plt.tight_layout()
plt.savefig('learning_curve_lgbm.png', dpi=150)
plt.show()

## 12. Save Results

In [ ]:
os.makedirs('../comparison', exist_ok=True)
results = pd.DataFrame([{
    'Algorithm':'LightGBM','Type':'Supervised',
    'Accuracy':round(accuracy,4),'Precision':round(precision,4),
    'Recall':round(recall,4),'F1 Score':round(f1,4),'ROC-AUC':round(roc_auc,4),
    'CV F1 Mean':round(cv_f1.mean(),4),'CV F1 Std':round(cv_f1.std(),4)
}])
print(results.to_string(index=False))
results.to_csv('../comparison/results_lightgbm.csv', index=False)
print('\nSaved: ../comparison/results_lightgbm.csv')